# Day 1: Infrastructure Foundation & Iceberg Storage Setup

This notebook simulates the local mock Iceberg catalog creating the Bronze raw payload schema with audit metadata.

In [ ]:
!pip install pyspark pyiceberg duckdb splink

In [ ]:
from pyspark.sql import SparkSession
import os

warehouse_path = os.path.join(os.getcwd(), "spark-warehouse")
os.makedirs(warehouse_path, exist_ok=True)

spark = SparkSession.builder \
    .appName("Day01_Storage_Setup") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", warehouse_path) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark session created with Iceberg configurations.")

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.ehdip_bronze")
print("Namespace local.ehdip_bronze created.")

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS local.ehdip_bronze.raw_payloads (
    ingestion_timestamp TIMESTAMP,
    source_system_id STRING,
    payload_id STRING,
    raw_payload_json STRING
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
""")
print("Bronze raw_payloads table created with audit metadata.")

In [ ]:
spark.sql("DESCRIBE TABLE local.ehdip_bronze.raw_payloads").show(truncate=False)